# E1/E4 — estratégias de seleção e robustez ao ruído · notebook de auditoria**Pergunta.** Qual estratégia de seleção ativa vence com oráculo perfeito(E1)? E o valor da seleção sobrevive quando o oráculo erra na taxa dos LLMsreais (E4)?**Este notebook difere do E0/E0-P: não é reanálise, é reexecução.** O`sweeps.jsonl` — as 104 células brutas (5 estratégias × 8 sementes + ablaçãode lote + E4: 3 níveis de ruído × 2 estratégias × 8 sementes) — **nunca foiversionado**; só sobreviveram os agregados (`analysis.json`, `baseline.json`).Achado do lote 2 da Etapa 2. Sem o bruto, não há reanálise possível: estenotebook roda `run_sweeps.py` de novo, do zero. CPU, ~30 min (medido: ~18 spor célula × 104).**Protocolo de dados: outro corte, específico deste experimento.** Pool de20.000 (amostrado da base deduplicada, semente 7) e teste de 5.000 — nem oCategorySchema de 621 do E0, nem a visão de 714 do populacional, nem o cortede P1/P2. Ver `sec:metodo-falco-baselines` no Cap. 3.**Regra desta auditoria.** Divergência é acusada, nunca corrigida.

In [ ]:
# 1) Onde estamos rodando.
import json, os, subprocess, sys
from pathlib import Path

def achar_raiz() -> Path:
    aqui = Path.cwd()
    for base in [aqui, *aqui.parents]:
        if (base / "experiments/e1e4").is_dir():
            return base
    destino = Path("/tmp/activelearning")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/GHDaru/activelearning.git", str(destino)],
                       check=True, capture_output=True)
    return destino

RAIZ = achar_raiz()
os.chdir(RAIZ)
RES = RAIZ / "experiments/e1e4/results"
print("raiz:", RAIZ)

In [ ]:
# 2) EXECUÇÃO — reexecução completa, não reanálise. Retomável: célula já
#    presente em sweeps.jsonl é pulada, então reexecutar após queda é seguro.
#    ~30 min em CPU (medido: ~18s/célula × 104). Não gasta cota de GPU.
import subprocess, sys, time

t0 = time.time()
proc = subprocess.Popen([sys.executable, "experiments/e1e4/run_sweeps.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in proc.stdout:
    print(linha, end="", flush=True)
rc = proc.wait()
assert rc == 0, f"run_sweeps.py terminou com código {rc}"
print(f"\ntotal: {(time.time()-t0)/60:.1f} min")

In [ ]:
# 3) E1 — as cinco estratégias, recalculadas do sweeps.jsonl (média ± desvio
#    em 8 sementes), confrontadas com a Tabela e1 do Cap. 5.
import statistics as est

sweeps = [json.loads(l) for l in (RES / "sweeps.jsonl").read_text().splitlines()]
e1 = [r for r in sweeps if r["exp"] == "e1"]

publicado = {
    "smallest_margin": (0.528, 0.013, 0.418, 0.013),
    "least_confidence": (0.518, 0.010, 0.421, 0.009),
    "entropy": (0.493, 0.006, 0.398, 0.008),
    "hybrid": (0.476, 0.014, 0.379, 0.008),
    "random": (0.444, 0.011, 0.339, 0.006),
}

print(f"{'estratégia':<18}{'n':>3}{'LCE rec':>10}{'F1 rec':>10}{'LCE pub':>10}{'F1 pub':>10}  veredito")
divergencias = []
for estrategia, (lce_p, lce_sd_p, f1_p, f1_sd_p) in publicado.items():
    execs = [r for r in e1 if r["strategy"] == estrategia]
    if len(execs) < 8:
        print(f"{estrategia:<18}{len(execs):>3}  incompleto — pule esta linha por ora")
        continue
    lce_m = est.mean(r["lce"] for r in execs)
    f1_m = est.mean(r["final_macro_f1"] for r in execs)
    ok = abs(lce_m - lce_p) < 0.02 and abs(f1_m - f1_p) < 0.02   # folga: nova amostragem do pool
    if not ok:
        divergencias.append((estrategia, lce_m, f1_m, lce_p, f1_p))
    print(f"{estrategia:<18}{len(execs):>3}{lce_m:>10.4f}{f1_m:>10.4f}"
          f"{lce_p:>10.3f}{f1_p:>10.3f}  {'OK' if ok else 'DIVERGE'}")

print(f"\ndivergências (folga 0,02 — o pool é reamostrado, não é o mesmo do "
      f"artefato original): {len(divergencias) or 'nenhuma'}")
print("nota: folga maior que nos outros notebooks porque run_sweeps.py monta o "
      "pool com random.Random(7).sample(...) a cada execução — não há garantia "
      "de que seja o MESMO pool de 20k do artefato original, só o mesmo desenho.")

In [ ]:
# 4) E4 — robustez ao ruído, mesma lógica.
e4 = [r for r in sweeps if r["exp"] == "e4"]
pub_e4 = {
    (0.1, "entropy"): 0.347, (0.1, "random"): 0.294,
    (0.2, "entropy"): 0.294, (0.2, "random"): 0.252,
    (0.4, "entropy"): 0.215, (0.4, "random"): 0.186,
}
print(f"{'ε':<6}{'estratégia':<12}{'n':>3}{'F1 rec':>10}{'F1 pub':>10}  veredito")
for (eps, estrategia), f1_p in pub_e4.items():
    execs = [r for r in e4 if r["noise"] == eps and r["strategy"] == estrategia]
    if len(execs) < 8:
        print(f"{eps:<6}{estrategia:<12}{len(execs):>3}  incompleto")
        continue
    f1_m = est.mean(r["final_macro_f1"] for r in execs)
    ok = abs(f1_m - f1_p) < 0.02
    print(f"{eps:<6}{estrategia:<12}{len(execs):>3}{f1_m:>10.4f}{f1_p:>10.3f}  "
          f"{'OK' if ok else 'DIVERGE'}")

In [ ]:
# 5) Gráfico: curvas de aprendizado da varredura E1 (uma semente por
#    estratégia, para não poluir — a tabela acima já deu a média das 8).
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

COR = {"entropy": "#2a78d6", "least_confidence": "#eb6834", "smallest_margin": "#1baf7a",
       "hybrid": "#eda100", "random": "#e87ba4"}
TINTA, TINTA2, GRADE = "#0b0b0b", "#52514e", "#d8d7d2"
vg = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))

plt.rcParams.update({"figure.dpi": 120, "font.size": 9,
                     "axes.edgecolor": GRADE, "axes.labelcolor": TINTA2,
                     "xtick.color": TINTA2, "ytick.color": TINTA2,
                     "axes.spines.top": False, "axes.spines.right": False})

fig, eixo = plt.subplots(figsize=(7.5, 4.3))
for estrategia, cor in COR.items():
    r = next((x for x in e1 if x["strategy"] == estrategia and x["seed"] == 0), None)
    if not r:
        continue
    xs = [p[0] for p in r["curve"]]
    ys = [p[1] for p in r["curve"]]
    eixo.plot(xs, ys, color=cor, linewidth=2, label=estrategia.replace("_", " "))
eixo.set_xlabel("rótulos gastos")
eixo.set_ylabel("Macro F1")
eixo.yaxis.set_major_formatter(vg)
eixo.set_title("E1 · curvas de aprendizado (semente 0)", color=TINTA, fontsize=11, loc="left")
eixo.grid(True, color=GRADE, linewidth=0.6, alpha=0.7)
eixo.set_axisbelow(True)
eixo.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## RodapéEste notebook **grava** `experiments/e1e4/results/sweeps.jsonl` — o artefatoque nunca existiu no repositório. **Use `git add -f`**: a linha 7 do`.gitignore` casa com ele.Para reexecutar do zero:```bashrm experiments/e1e4/results/sweeps.jsonlpython experiments/e1e4/run_sweeps.py   # ~30 min, retomável```